# Football Tactical AI - YOLO11 4-Class Training\n\nPrimary model: `yolo11l.pt` for high-quality market baseline.\n\nClasses:\n\n```text\n0 player\n1 goalkeeper\n2 referee\n3 ball\n```

In [ ]:
!nvidia-smi\n!pip install -U ultralytics roboflow

## Option A - Upload YOLO Dataset ZIP\n\nUpload a YOLO-format dataset zip containing `data.yaml`, `images/`, and `labels/`.

In [ ]:
from google.colab import files\nuploaded = files.upload()\n\nimport zipfile\nfrom pathlib import Path\n\nDATASET_ROOT = Path('/content/football_yolo11_dataset')\nDATASET_ROOT.mkdir(parents=True, exist_ok=True)\n\nfor filename in uploaded:\n    if filename.endswith('.zip'):\n        with zipfile.ZipFile(filename) as zf:\n            zf.extractall(DATASET_ROOT)\n\nprint('Dataset files:')\n!find /content/football_yolo11_dataset -maxdepth 3 -type f | head -50

## Option B - Roboflow Export\n\nUse this instead of the upload cell if your annotations are in Roboflow.

In [ ]:
# from roboflow import Roboflow\n# rf = Roboflow(api_key='YOUR_API_KEY')\n# project = rf.workspace('YOUR_WORKSPACE').project('YOUR_PROJECT')\n# version = project.version(1)\n# dataset = version.download('yolov11')\n# DATA_YAML = dataset.location + '/data.yaml'

## Verify Dataset

In [ ]:
from pathlib import Path\nDATA_YAML = '/content/football_yolo11_dataset/data.yaml'\nassert Path(DATA_YAML).exists(), f'Missing data.yaml: {DATA_YAML}'\n!cat /content/football_yolo11_dataset/data.yaml\n!find /content/football_yolo11_dataset/images -type f | wc -l\n!find /content/football_yolo11_dataset/labels -type f | wc -l

## Train YOLO11\n\nStart with `yolo11l.pt`. If memory fails, switch to `yolo11m.pt` or lower `imgsz`.

In [ ]:
from ultralytics import YOLO\n\nMODEL = 'yolo11l.pt'\nRUN_NAME = 'yolo11l_football_4class_v1'\n\nmodel = YOLO(MODEL)\nresults = model.train(\n    data=DATA_YAML,\n    epochs=120,\n    imgsz=1280,\n    batch=-1,\n    patience=30,\n    cos_lr=True,\n    close_mosaic=15,\n    device=0,\n    project='/content/runs/football_yolo11',\n    name=RUN_NAME,\n)

## Validate And Export

In [ ]:
best_model = f'/content/runs/football_yolo11/{RUN_NAME}/weights/best.pt'\nmodel = YOLO(best_model)\nmetrics = model.val(data=DATA_YAML, imgsz=1280, device=0)\nprint(metrics)\n\nexport_path = model.export(format='onnx', imgsz=1280)\nprint('best.pt:', best_model)\nprint('onnx:', export_path)

## Download Artifacts

In [ ]:
import shutil\nfrom google.colab import files\n\nartifact_dir = f'/content/runs/football_yolo11/{RUN_NAME}'\nzip_path = f'/content/{RUN_NAME}_artifacts.zip'\nshutil.make_archive(zip_path.replace('.zip', ''), 'zip', artifact_dir)\nfiles.download(zip_path)